### Prepare environment

In [0]:
%run ../environment/prepare_environment

# Convolutional Neural Networks - Feature Extraction & Machine's Eyes

##Introduction
Convolutional Neural Network is a more advanced NN example, especially useful in image processing tasks. In this notebook we will train a CNN on MNIST and visualize feature maps.

This notebook will cover:
- Understanding convolutional neural networks (CNNs) for image classification
- How convolutional filters detect local features (edges, textures)
- Pooling for spatial dimensionality reduction
- Training a small CNN on MNIST
- Visualizing learned feature maps to interpret model decisions

## 1. Load and preprocess data

Unlike dense networks that flatten images, CNNs preserve **spatial structure** using convolutional filters. Each filter slides over the image, computing dot products at each position.

We will start with loading the dataset and normalizing pixels to `[0,1]`. We also need to reshape the images - MNIST images need to be (28, 28, 1) for Conv2D, where 1 is the channel count (grayscale). RGB images would have 3 channels. This 4D shape (batch, height, width, channels) allows the framework to apply 2D operations correctly.

In [0]:
import os
import random
import numpy as np
import tensorflow as tf
import mlflow
import mlflow.tensorflow
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

seed = 89
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]
print('x_train', x_train.shape, 'x_test', x_test.shape)

## 2. Build CNN model

A simple CNN architecture we wil build follows this pattern:
- **Conv2D(8, 3x3, ReLU)**: 8 filters learn to detect initial features like edges and corners. Using fewer filters here significantly reduces the initial parameter count.
- **MaxPooling2D(2x2)**: Reduces spatial dimensions (26×26→13×13), making the model robust to small translations and reducing computation.
- **Conv2D(16, 3x3, ReLU)**: 16 filters learn more complex patterns (like loops or intersections) by combining the features from the first layer.
- **MaxPooling2D(2x2)**: Further reduces dimensions (11×11→5×5), ensuring the final vector is compact and focused.
- **Flatten**: Converts the small 2D feature maps (5x5) into a 1D vector of 400 elements.
- **Dense layers**: Perform the final classification. Because the CNN layers have already extracted high-level features, these layers can be much smaller (e.g., 32 units) than in a standard MLP.

The MaxPooling selects the strongest activation in each 2x2 window, preserving important features while reducing noise. This also prevents overfitting by enforcing invariance to small shifts.

Similar to MLPs, we train with backpropagation. However, CNNs learn **sparse, hierarchical representations**. Early layers typically detect simple features (edges), while later layers combine them into complex patterns (shapes, objects).

CNNs generalize better to new images than MLPs because they exploit locality: nearby pixels are related, so sharing weights across positions is efficient.

In [0]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)), 
    layers.Conv2D(8, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(16, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

model.summary()

## 3. Train and log

CNNs train similarly to MLPs but with a key difference: **weight sharing**. Filters are reused across the image, so the model learns ONE set of edge detectors that apply everywhere. This drastically reduces parameters compared to dense layers.

Training dynamics:
- **Convolutional layers**: Learn hierarchical features (edges → shapes → objects)
- **Pooling layers**: Enforce spatial invariance (robustness to small shifts)
- **Dense layers**: Combine features for classification

CNNs typically need fewer epochs than MLPs to converge because spatial structure is preserved. However, they may require more data to learn reliable filters. The validation curve helps decide when to stop (early stopping).

In [0]:
mlflow.tensorflow.autolog(
    silent=True,
    registered_model_name='ai_ml_in_practice.neural_networks_silver.mnist_cnn_model'
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

with mlflow.start_run(run_name='cnn_mnist') as run:
    history = model.fit(
        x_train, 
        y_train, 
        epochs=50,
        batch_size=32, 
        validation_split=0.2, 
        callbacks=[early_stop],
        verbose=2
    )
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    
    mlflow.log_metrics({
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss)
    })

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title('CNN Accuracy')
    ax1.legend()
    
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title('CNN Loss')
    ax2.legend()
    
    mlflow.log_figure(fig, "cnn_training_summary.png")
    plt.show()

print(f"Run complete. Model registered as 'mnist_cnn_model'.")

## 4. Visualize feature maps

Feature maps show the activation of each filter for a specific input image. Bright areas indicate where a filter detected its learned feature. By visualizing these, we gain insight into the hierarchical way the model "sees" and interprets data.

In our multi-layer architecture, feature map visualization reveals a clear progression of complexity:
- First Conv2D Layer (8 filters): These filters act as low-level detectors. For example for digit 0, the resolution is high `(26×26)`, and each filter specializes in a different direction:
    - Vertical Edges: Notice how some maps (like the 4th and 7th) highlight only the left or right "walls" of the zero.
    - Horizontal Edges: Other maps (like the 2nd) focus on the top and bottom curves.
    - Outline vs. Texture: Some filters capture the entire silhouette, while others ignore the center to focus purely on the stroke.
- Second Conv2D Layer (16 filters): After pooling, these maps become smaller and more abstract. They stop looking like a "0" and start looking like localized heatmaps. They look for:
    - Structural Components: Specific arcs or the "hollow" center of the zero.
    - Feature Combinations: Combining the simple lines from the first layer into a closed loop.

Why are feature maps important?
- **Explainability**: We can verify that for a "0", the model is actually looking at the circular shape and not just a random cluster of pixels in the corner.
- **Debugging**: If one of these 8 maps was completely dark, it would be a "dead filter," suggesting the model might be too large for the task or trained inefficiently.
- **Architectural Insight**: It might act as a proof the CNN's advantage over MLP. While an MLP sees 784 independent pixels, these maps prove the CNN understands spatial relationships (how pixels form a curve).

Limitations:
Deeper layers' activations are harder to interpret. In a complex model, a "0" might activate a specific combination of filters that doesn't look like a circle to us, but represents "circularity" to the computer. This is why tools like Grad-CAM or Attention Maps are essential for deep learning research.

In [0]:
loaded_model = mlflow.tensorflow.load_model(
    "models:/ai_ml_in_practice.neural_networks_silver.mnist_cnn_model/1"
)

In [0]:
target_digit = 0

digit_indices = np.where(y_test == target_digit)[0]
idx = random.choice(digit_indices)
x_digit = x_test[idx:idx+1]

# Original image
plt.figure(figsize=(3,3))
plt.imshow(x_digit[0, :, :, 0], cmap='gray')
plt.title(f'Input: Random Digit {target_digit}')
plt.axis('off')
plt.show()

# Feature maps
conv_layers = [layer for layer in loaded_model.layers if isinstance(layer, keras.layers.Conv2D)]

for layer in conv_layers:
    # Auxilary model extracting the outputs from each Conv2D layer
    intermediate_model = keras.Model(inputs=loaded_model.layers[0].input, outputs=layer.output)
    feat_maps = intermediate_model.predict(x_digit, verbose=0)
    
    num_filters = feat_maps.shape[-1]
    
    # Dynamic grid size
    cols = 8 if num_filters >= 8 else 4
    rows = (num_filters + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols*1.5, rows*1.5))
    fig.suptitle(f'Layer: {layer.name} | Shape: {feat_maps.shape[1:3]} | Filters: {num_filters}', fontsize=14)
    
    for i in range(rows * cols):
        if rows > 1:
            ax = axes.flat[i]
        else:
            ax = axes[i] if cols > 1 else axes

        if i < num_filters:
            ax.imshow(feat_maps[0, :, :, i], cmap='viridis')
        ax.axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.85)
    plt.show()